# Week 5: Topic-model diagnostics, robustness and validity


Week 4 fitted one LDA and one NMF model and showed that they broadly agree. This notebook asks whether those topics, and the institutional differences built on them, can be trusted. Each section answers one diagnostic question:

| § | Diagnostic | Question | Why it is appropriate for topic models |
|---|---|---|---|
| 1 | Seed stability | Refit with different random seeds: do the same topics come back? | LDA/NMF optimise a non-convex objective, so one run can land in a local optimum |
| 2 | K sensitivity | Do the key topics persist, split or merge at K = 20 | K was chosen by a coherence proxy, and coherence differences between K were small |
| 3 | Preprocessing sensitivity | Do topics survive other reasonable preprocessing choices? | Week 4 admitted the topic route had an extra boilerplate pass the other two routes lacked |
| 4 | Validation sample | Draws the stratified ~200-paragraph hand-coding sample (blind coding sheets for both coders) | Nothing so far shows that a topic reads the way its label says |
| 5 | Validation analysis | Inter-coder κ, and topic-derived frame versus the gold labels | Run **after** both coding sheets are filled in |
| 6 | Figure and summary | Figure panel (b) and a table of numbers for the report | |


In [2]:
from __future__ import annotations

import json
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import cohen_kappa_score, classification_report

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # random-init NMF refits

# ---- run configuration -----------------------------------------------------
FAST = False                      # True = fewer refits, for a quick test run
RANDOM_STATE = 0                  
K_REF = 10                        
N_SEEDS = 6                        
LDA_ITER = 10 if FAST else 25     # Week 4 used 25
MATCH_THRESHOLD = 0.8             # cosine above which a refit topic counts as "the same topic"
N_TRACK = 4                       # number of institution-distinguishing topics tracked in the figure
VALIDATION_N = {"Commission": 90, "Parliament": 55, "Council": 55}   # ~200, Council and EP oversampled

OUT_DIR = Path("reports/week5_topic_diagnostics"); OUT_DIR.mkdir(parents=True, exist_ok=True)
VALID_DIR = Path("data/validation"); VALID_DIR.mkdir(parents=True, exist_ok=True)

INSTITUTIONS = ["Commission", "Council", "Parliament"]
INST_COLOURS = {"Commission": "#2a78d6", "Council": "#eb6834", "Parliament": "#1baf7a"}

SUMMARY: dict = {}
T0 = time.time()

## Load the corpus

In [3]:
CANDIDATES = [
    Path("data/corpus/paragraphs.parquet"),
    Path("../data/corpus/paragraphs.parquet"),
    Path("paragraphs.parquet"),
    Path("/mnt/user-data/uploads/paragraphs.parquet"),
]
CORPUS_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CORPUS_PATH is None:
    raise FileNotFoundError("Could not find paragraphs.parquet - edit CANDIDATES.")

raw = pd.read_parquet(CORPUS_PATH)
if "para_idx" not in raw.columns:                       # stable paragraph key
    raw["para_idx"] = raw.groupby("doc_id").cumcount()
for col in ["doc_type", "genre"]:
    if col not in raw.columns:
        raw[col] = "unknown"
print(f"paragraphs: {len(raw)} | documents: {raw.doc_id.nunique()}")
print(raw.groupby("institution").agg(documents=("doc_id", "nunique"), paragraphs=("text", "size")))

paragraphs: 27732 | documents: 139
             documents  paragraphs
institution                       
Commission          75       17695
Council             28        4819
Parliament          36        5218


## 0. Preprocessing and the reference model

`prepare()` reproduces Week 4 exactly: lowercase, alphabetic tokens of 3+ characters, English and EU-procedural stopwords, drop figure/table captions, drop paragraphs with fewer than 8 tokens left. The reference model is Week 4's specification (K = 10, seed 0).

In [5]:
DOMAIN_STOPWORDS = {
    "article", "articles", "paragraph", "paragraphs", "regulation", "regulations",
    "directive", "directives", "shall", "member", "members", "states", "state",
    "union", "european", "commission", "council", "parliament", "annex",
    "recital", "whereas", "pursuant", "hereby", "eu", "eus", "proposal",
}
ELI_STOPWORDS = {"eli", "http", "https", "europa"}      # Week 4's extra boilerplate pass
# Institutional rhetorical verbs: the third stopword list in Tim's Week 4 pipeline.
RHETORICAL_STOPWORDS = {
    "calls", "call", "stresses", "stress", "underlines", "highlights", "notes", "note",
    "welcomes", "recalls", "emphasises", "emphasizes", "considers", "believes",
    "regrets", "urges", "invites", "encourages", "insists", "acknowledges",
    "recognises", "reiterates", "points", "having", "regard", "whereas",
}
TOKEN = re.compile(r"[a-zA-Z]{3,}")
CAPTION_PATTERN = re.compile(r"(?i)^\s*(figure|table|chart)\s+\d+\s*[:.]")
BASE_VEC_KWARGS = dict(min_df=5, max_df=0.5, ngram_range=(1, 2), max_features=4000)


def prepare(df, *, eli=True, captions=True, rhetorical=False, min_tokens=8):
    stop = set(ENGLISH_STOP_WORDS) | DOMAIN_STOPWORDS
    if eli:
        stop |= ELI_STOPWORDS
    if rhetorical:
        stop |= RHETORICAL_STOPWORDS
    out = df.copy()
    if captions:
        out = out[~out["text"].str.match(CAPTION_PATTERN)]
    out["text_proc"] = out["text"].map(
        lambda t: " ".join(w for w in TOKEN.findall(t.lower()) if w not in stop))
    out = out[out["text_proc"].str.split().map(len) >= min_tokens]
    return out.reset_index(drop=True)


def vectorise(texts, kind, **overrides):
    kw = {**BASE_VEC_KWARGS, **overrides}
    vec = (CountVectorizer if kind == "count" else TfidfVectorizer)(**kw)
    X = vec.fit_transform(texts)
    return X, np.array(vec.get_feature_names_out())


def fit_topics(X, model, k, seed, nmf_init="nndsvda"):
    # Returns (topic-word matrix with rows summing to 1, paragraph-topic matrix with rows summing to 1).
    if model == "LDA":
        m = LatentDirichletAllocation(n_components=k, random_state=seed, max_iter=LDA_ITER,
                                      learning_method="online", batch_size=2048)
    else:
        m = NMF(n_components=k, random_state=seed, init=nmf_init,
                max_iter=300 if nmf_init == "nndsvda" else 600)
    theta = m.fit_transform(X)
    theta = theta / np.clip(theta.sum(axis=1, keepdims=True), 1e-12, None)
    phi = m.components_ / m.components_.sum(axis=1, keepdims=True)
    return phi, theta


def top_words(phi, vocab, n=10):
    return [", ".join(vocab[np.argsort(-row)[:n]]) for row in phi]


def cosine_matrix(phi_a, vocab_a, phi_b, vocab_b):
    # Cosine similarity between every topic of model A and of model B, on their shared vocabulary.
    common = np.intersect1d(vocab_a, vocab_b)
    ia = pd.Index(vocab_a).get_indexer(common)
    ib = pd.Index(vocab_b).get_indexer(common)
    A, B = phi_a[:, ia], phi_b[:, ib]
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A @ B.T


def match(phi_ref, vocab_ref, phi_new, vocab_new):
    # One-to-one Hungarian matching of new topics onto reference topics (same K).
    # Returns perm (perm[i] = new topic matched to ref topic i) and the cosine of each pair.
    S = cosine_matrix(phi_ref, vocab_ref, phi_new, vocab_new)
    rows, cols = linear_sum_assignment(-S)
    perm = np.empty(len(rows), dtype=int)
    perm[rows] = cols
    return perm, S[rows, cols][np.argsort(rows)]


def profile(theta, meta, weighting="paragraph"):
    # Institution x topic prevalence, rows normalised to 1 (as in Week 4).
    cols = [f"T{t}" for t in range(theta.shape[1])]
    df = pd.DataFrame(theta, columns=cols)
    df["institution"] = meta["institution"].values
    df["doc_id"] = meta["doc_id"].values
    if weighting == "document":
        df = df.groupby(["institution", "doc_id"])[cols].mean().reset_index()
    tab = df.groupby("institution")[cols].mean()
    return tab.div(tab.sum(axis=1), axis=0).reindex(INSTITUTIONS)


paras = prepare(raw)
X_counts, vocab_c = vectorise(paras["text_proc"], "count")
X_tfidf, vocab_t = vectorise(paras["text_proc"], "tfidf")
print(f"paragraphs modelled: {len(paras)} | vocabulary: {len(vocab_c)}")

REF = {}
for model, X, vocab in [("LDA", X_counts, vocab_c), ("NMF", X_tfidf, vocab_t)]:
    phi, theta = fit_topics(X, model, K_REF, RANDOM_STATE)
    REF[model] = dict(phi=phi, theta=theta, vocab=vocab, X=X, words=top_words(phi, vocab))
    print(f"\n--- reference {model} (K={K_REF}, seed {RANDOM_STATE}) ---")
    for t, w in enumerate(REF[model]["words"]):
        print(f"  T{t}: {w}")

ref_tables = []
for model in REF:
    prof = profile(REF[model]["theta"], paras).T
    prof.insert(0, "top_words", REF[model]["words"])
    prof.insert(0, "model", model)
    ref_tables.append(prof)
pd.concat(ref_tables).to_csv(OUT_DIR / "reference_topics.csv")

paragraphs modelled: 25904 | vocabulary: 4000

--- reference LDA (K=10, seed 0) ---
  T0: liability, law, justice, having, regard, rules, damage, having regard, product, framework
  T1: data, online, services, protection, content, act, users, dma, use, personal
  T2: digital, europe, support, smes, programme, innovation, skills, national, funding, including
  T3: systems, risk, high risk, high, assessment, legislation, conformity, risk systems, harmonisation, conformity assessment
  T4: systems, authorities, risk, national, market, rights, high, fundamental, high risk, ensure
  T5: systems, education, training, learning, human, cybersecurity, development, use, models, including
  T6: intelligence, artificial, artificial intelligence, acts, requirements, set, chapter, systems, safety, delegated
  T7: data, digital, public, market, services, energy, development, regulatory, cross, single
  T8: liability, services, legal, costs, cloud, burden, rules, specific, products, market
  T9: data,

**Which topics to track.** The Week 4 argument rests on topics that separate the institutions (e.g. the Council's enforcement/compliance and high-risk conformity topics). For each model, the tracked topics are the N_TRACK topics with the largest spread between the highest and lowest institution share. Check that they are the substantively important ones; to track a different set, overwrite `TRACK` by hand.

In [6]:
TRACK = {}
for model in REF:
    prof = profile(REF[model]["theta"], paras)
    spread = (prof.max() - prof.min()).sort_values(ascending=False)
    TRACK[model] = [int(c[1:]) for c in spread.index[:N_TRACK]]
    print(f"\n{model}: tracked topics")
    for t in TRACK[model]:
        lead = prof[f"T{t}"].idxmax()
        print(f"  T{t} (leader: {lead} {prof[f'T{t}'].max():.3f}): {REF[model]['words'][t][:70]}")

TRACK["LDA"] = [0, 3, 5, 2]  


LDA: tracked topics
  T4 (leader: Council 0.351): systems, authorities, risk, national, market, rights, high, fundamenta
  T3 (leader: Council 0.184): systems, risk, high risk, high, assessment, legislation, conformity, r
  T7 (leader: Parliament 0.163): data, digital, public, market, services, energy, development, regulato
  T8 (leader: Commission 0.130): liability, services, legal, costs, cloud, burden, rules, specific, pro

NMF: tracked topics
  T4 (leader: Council 0.241): authorities, national, competent, authority, competent authorities, of
  T0 (leader: Commission 0.228): technologies, development, support, use, research, sector, innovation,
  T1 (leader: Council 0.173): risk, high risk, high, systems, risk systems, providers, requirements,
  T5 (leader: Commission 0.148): liability, rules, damage, legal, liability rules, burden, proof, produ


## 1. Seed stability

Refit each model with `N_SEEDS` different seeds, match each refit's topics one-to-one onto the reference topics (Hungarian matching on topic-word cosine), and record (a) how similar each matched topic is and (b) the institution shares of the matched topics.

NMF with `nndsvda` initialisation is deterministic (the random seed is effectively unused), so it would look perfectly stable by construction. For an honest test, the NMF refits use `init="random"`.

**How to read the output:** a topic whose mean cosine is ≥ 0.8 in almost every run is a stable topic. A topic that often falls below that is a product of one particular run and should not carry an interpretation. The institution-share columns show how far the Week 4 numbers move from run to run.

In [7]:
def refit_series(label, fits):
    # fits: iterable of (run_id, model, phi, theta, vocab, meta). Returns tidy records.
    sims, shares = [], []
    for run_id, model, phi, theta, vocab, meta in fits:
        perm, cos = match(REF[model]["phi"], REF[model]["vocab"], phi, vocab)
        prof = profile(theta[:, perm], meta)
        for t in range(K_REF):
            sims.append(dict(check=label, run=run_id, model=model, topic=t, cosine=cos[t]))
            for inst in INSTITUTIONS:
                shares.append(dict(check=label, run=run_id, model=model, topic=t,
                                   institution=inst, share=prof.loc[inst, f"T{t}"]))
    return pd.DataFrame(sims), pd.DataFrame(shares)


def seed_fits():
    for s in range(1, N_SEEDS + 1):
        for model in REF:
            init = "random" if model == "NMF" else None
            phi, theta = fit_topics(REF[model]["X"], model, K_REF, s, nmf_init=init or "nndsvda")
            yield s, model, phi, theta, REF[model]["vocab"], paras
        print(f"  seed {s} done ({time.time() - T0:.0f}s)")


seed_sims, seed_shares = refit_series("seed", seed_fits())


def stability_table(sims):
    g = sims.groupby(["model", "topic"])["cosine"]
    tab = pd.DataFrame({"mean_cos": g.mean(), "min_cos": g.min(),
                        "share_runs_matched": g.apply(lambda c: (c >= MATCH_THRESHOLD).mean())})
    tab["top_words"] = [REF[m]["words"][t][:60] for m, t in tab.index]
    return tab.round(3)


seed_tab = stability_table(seed_sims)
print(seed_tab.to_string())
seed_tab.to_csv(OUT_DIR / "stability_seed.csv")

share_sd = (seed_shares.groupby(["model", "topic", "institution"])["share"]
            .agg(["mean", "std", "min", "max"]).round(3))
print("\nInstitution shares across seeds (tracked topics):")
for model in REF:
    print(share_sd.loc[model].loc[TRACK[model]].to_string())
share_sd.to_csv(OUT_DIR / "stability_seed_shares.csv")

for model in REF:
    SUMMARY[f"seed_{model}_n_stable_topics"] = int((seed_tab.loc[model, "share_runs_matched"] >= 0.8).sum())
    SUMMARY[f"seed_{model}_mean_cosine"] = float(seed_tab.loc[model, "mean_cos"].mean().round(3))

  seed 1 done (728s)
  seed 2 done (841s)
  seed 3 done (955s)
  seed 4 done (1064s)
  seed 5 done (1179s)
  seed 6 done (1287s)
             mean_cos  min_cos  share_runs_matched                                                     top_words
model topic                                                                                                     
LDA   0         0.319    0.116               0.000  liability, law, justice, having, regard, rules, damage, havi
      1         0.624    0.516               0.000  data, online, services, protection, content, act, users, dma
      2         0.832    0.770               0.667  digital, europe, support, smes, programme, innovation, skill
      3         0.864    0.706               0.833  systems, risk, high risk, high, assessment, legislation, con
      4         0.776    0.642               0.333  systems, authorities, risk, national, market, rights, high, 
      5         0.571    0.257               0.167  systems, education, training

## 2. K sensitivity

**Why this value of K.** Week 4 chose K = 10 because it had the highest coherence in {5, 8, 10, 12}. Coherence-maximisation is known to favour too few topics, and there is no objectively correct K, so K = 10 is kept as the reference only for continuity with Week 4. This section tests one alternative, K = 20, to check whether the tracked topics and their leading institution survive a finer-grained decomposition.

**What is measured.** Since K = 10 and K = 20 cannot be matched one-to-one, each K = 20 topic is assigned to the K = 10 reference topic it most resembles (its "parent"):

- `family_cos`: the cosine between a reference topic and the prevalence-weighted sum of its children at K = 20. A topic that **splits** into two or three sub-themes at K = 20 still scores high here — that is the expected, reassuring outcome if the pieces belong to the same underlying frame.
- `same_leader`: whether the institution leading a topic's family at K = 20 is the same as at K = 10. This is the test that matters for the argument: the claim is about which institution emphasises what, not about the exact number of topics.

In [8]:
K_ALT = [20] 

In [9]:
k_rows = []
for k in K_ALT:
    for model in REF:
        phi, theta = fit_topics(REF[model]["X"], model, k, RANDOM_STATE)
        vocab = REF[model]["vocab"]
        S = cosine_matrix(REF[model]["phi"], vocab, phi, vocab)          # ref x alt
        parent = S.argmax(axis=0)                                         # parent of each alt topic
        prevalence = theta.mean(axis=0)
        alt_prof = profile(theta, paras)                                  # institution x alt topic
        ref_prof = profile(REF[model]["theta"], paras)
        alt_words = top_words(phi, vocab, n=6)
        for t in range(K_REF):
            children = np.where(parent == t)[0]
            best = int(S[t].argmax())
            if len(children):
                fam = (phi[children] * prevalence[children, None]).sum(axis=0)
                fam_cos = float(fam @ REF[model]["phi"][t] /
                                (np.linalg.norm(fam) * np.linalg.norm(REF[model]["phi"][t])))
                fam_share = alt_prof[[f"T{j}" for j in children]].sum(axis=1)
            else:
                fam_cos = np.nan
                fam_share = alt_prof[f"T{best}"]
            k_rows.append(dict(
                model=model, K=k, topic=t, best_cos=S[t, best], family_cos=fam_cos,
                n_children=len(children), merged=len(children) == 0,
                ref_leader=ref_prof[f"T{t}"].idxmax(), alt_leader=fam_share.idxmax(),
                children_words=" | ".join(alt_words[j] for j in children) if len(children) else f"(merged into: {alt_words[best]})",
            ))
    print(f"  K={k} done ({time.time() - T0:.0f}s)")
k_tab = pd.DataFrame(k_rows)
k_tab["same_leader"] = k_tab.ref_leader == k_tab.alt_leader
k_tab["recovered_cos"] = k_tab.family_cos.fillna(k_tab.best_cos)   # used in the figure
k_tab = k_tab.round(3)
k_tab.to_csv(OUT_DIR / "stability_K.csv", index=False)

print(k_tab.pivot_table(index=["model", "topic"], columns="K",
                        values=["recovered_cos", "n_children", "same_leader"], aggfunc="first").to_string())
print("\nTracked topics: does the leading institution survive a change of K?")
for model in REF:
    sub = k_tab[(k_tab.model == model) & k_tab.topic.isin(TRACK[model])]
    print(f"\n{model}")
    print(sub.pivot_table(index=["topic", "ref_leader"], columns="K", values="alt_leader", aggfunc="first").to_string())
for model in REF:
    for k in K_ALT:
        sub = k_tab[(k_tab.model == model) & (k_tab.K == k)]
        SUMMARY[f"K{k}_{model}_n_recovered_topics"] = int((sub.recovered_cos >= MATCH_THRESHOLD).sum())
        SUMMARY[f"K{k}_{model}_n_merged_topics"] = int(sub.merged.sum())
        SUMMARY[f"K{k}_{model}_tracked_same_leader"] = int(sub[sub.topic.isin(TRACK[model])].same_leader.sum())

print("\nChildren of each tracked topic at the largest K (read these to judge whether a split is sub-themes of one frame):")
for model in REF:
    sub = k_tab[(k_tab.model == model) & (k_tab.K == max(K_ALT)) & k_tab.topic.isin(TRACK[model])]
    for _, r in sub.iterrows():
        print(f"  {model} T{r.topic}: {r.children_words}")

  K=20 done (1470s)
            n_children recovered_cos same_leader
K                   20            20          20
model topic                                     
LDA   0              1         0.792        True
      1              1         0.730        True
      2              3         0.921        True
      3              2         0.912        True
      4              2         0.864        True
      5              2         0.749       False
      6              1         0.899       False
      7              2         0.877       False
      8              3         0.908        True
      9              3         0.855        True
NMF   0              5         0.753        True
      1              1         0.985        True
      2              2         0.888        True
      3              2         0.949       False
      4              3         0.894        True
      5              1         0.986        True
      6              2         0.931        True
